## CHEBII PAMELA JEPKORIR
## **Consumer Behavior Analytics Using Transaction Data**

In [4]:
import pandas as pd
df = pd.read_csv("online_retail_II.csv")
df.tail()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,12/9/2011 12:50,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,12/9/2011 12:50,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,12/9/2011 12:50,4.15,12680.0,France
541908,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,12/9/2011 12:50,4.95,12680.0,France
541909,581587,POST,POSTAGE,1,12/9/2011 12:50,18.00,12680.0,France


### 1. BASIC INSPECTION
- Here we check the shape of the dataset to know the number of rows and colums
- Here we also check the information of the dataset to know the exact columns and the data types of the columns
- We also check whether the dataset has missing values

In [6]:
df.shape
df.info()
df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      541910 non-null  object 
 1   StockCode    541910 non-null  object 
 2   Description  540456 non-null  object 
 3   Quantity     541910 non-null  int64  
 4   InvoiceDate  541910 non-null  object 
 5   Price        541910 non-null  float64
 6   Customer ID  406830 non-null  float64
 7   Country      541910 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


Invoice             0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
Price               0
Customer ID    135080
Country             0
dtype: int64

- The customer ID and Description has 135,080 and 1454 missing values respectively

## **EXPLORATORY DATA ANALYSIS**

### 1. Cleaning the dataset

#### a) Remove Missing Customer IDs
- Here all rows with missing customer IDs are dropped

In [8]:
df = df.dropna(subset=['Customer ID'])
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 406830 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      406830 non-null  object 
 1   StockCode    406830 non-null  object 
 2   Description  406830 non-null  object 
 3   Quantity     406830 non-null  int64  
 4   InvoiceDate  406830 non-null  object 
 5   Price        406830 non-null  float64
 6   Customer ID  406830 non-null  float64
 7   Country      406830 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 27.9+ MB


### 2. **Identification of Cancelled Transactions**
#### a) Inspect Unique Invoice Patterns

In [9]:
df['Invoice'].astype(str).str[:1].value_counts()

Invoice
5    397925
C      8905
Name: count, dtype: int64

- A subset of invoice numbers begins with the letter 'C', indicating a different transaction type from standard numeric invoices.

#### b)  Compare Quantities for Those Invoices
- Check whether these rows contain negative quantities

In [10]:
df[df['Invoice'].astype(str).str.startswith('C')]['Quantity'].describe()

count     8905.000000
mean       -30.859966
std       1170.154939
min     -80995.000000
25%         -6.000000
50%         -2.000000
75%         -1.000000
max         -1.000000
Name: Quantity, dtype: float64

- The invoices with Cancellations have negative quantities
- Comparing it with normal invoces;

In [11]:
df[~df['Invoice'].astype(str).str.startswith('C')]['Quantity'].describe()

count    397925.000000
mean         13.021793
std         180.419984
min           1.000000
25%           2.000000
50%           6.000000
75%          12.000000
max       80995.000000
Name: Quantity, dtype: float64

- The observation show that normal purchases have positive values
- This confirms that invoices starting with 'C' correspond to product returns or cancellations.

#### c) Show Monetary Impact of 'C' Invoices

In [12]:
df[df['Invoice'].astype(str).str.startswith('C')]['Quantity'].sum()

-274808

- The total is negative, further confirming these are returns.

### **Identification of Cancelled Transactions Summary**

After removing records with missing Customer IDs, invoice identifiers were analyzed to understand transaction structure. It was observed that 397,925 invoices began with numeric values, while 8,905 invoices began with the letter 'C'.

Further investigation revealed that invoices beginning with “C” were associated with negative quantities, indicating product returns or cancelled transactions. Since the objective of this study is to analyze actual purchasing behavior and revenue generation, cancelled transactions were excluded from the dataset.

This preprocessing step ensures that customer segmentation, churn prediction, and lifetime value estimation are based solely on completed purchase behavior.


### 3. Removing Negative Quantities
- This step ensures that all negative quantities, including cancellations, are revoved.

In [16]:
df = df[df['Quantity'] > 0]
df.shape

(397925, 8)

### 4. Creating Total Revenue Column

In [17]:
df['TotalPrice'] = df['Quantity'] * df['Price']
df.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country', 'TotalPrice'],
      dtype='object')

To quantify the monetary contribution of each transaction, a new variable (TotalPrice) was created by multiplying the quantity purchased by the unit price. This transformation enables accurate computation of customer-level revenue metrics, which are essential for segmentation, churn modeling, and lifetime value estimation.

### 